In [1]:
from sklearn.preprocessing import LabelEncoder
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset
from transformers import ASTForAudioClassification
from transformers import ASTModel
from transformers import DefaultDataCollator
#from datasets import load_metric
import evaluate
from transformers import Trainer, TrainingArguments
import os
from transformers import EarlyStoppingCallback

c:\Users\Kochana\projects\genres\ast_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0515 19:07:05.356000 3900 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
import os 
from huggingface_hub import login
with open (r"C:\Users\Kochana\projects\hf_token.txt") as f:
    token = f.read()
    login(token = token)

In [3]:
data_path = Path(r"C:\Users\Kochana\projects\genres\data\gtzan\gtzan.npz")
data = np.load(data_path)
lst = data.files

In [24]:
tracks_path = []
labels = []
#data_path = Path(r"/content/drive/MyDrive/data/gtzan_old")
for path in lst:
    #path_file = os.path.join(data_path, file, "track.npy")
    tracks_path.append(path)
    labels.append(path[6:][:-12])
le = LabelEncoder()
encoded_labels = le.fit_transform(labels)
train, test, train_labels, test_labels = train_test_split(
        tracks_path, encoded_labels, test_size=0.1, stratify=encoded_labels, random_state=42)
train, validation, train_labels, validation_labels = train_test_split(
        train, train_labels, test_size=0.2, stratify=train_labels, random_state=42)

In [30]:
class GTZANSpectrogramDataset(Dataset):
    def __init__(self, path, labels):
        self.paths = path
        self.labels = labels
        self.max_time = 1020
        self.data = data
        
    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        #spec = self.spectrograms[idx]  # shape: (128, time)
        spec = data[self.paths[idx]]
        if spec.ndim == 3 and spec.shape[0] == 1:
            spec = spec.squeeze(0)
        spec = spec[:self.max_time, :]
        spec = torch.tensor(spec, dtype=torch.float32)

        #spec = spec.unsqueeze(0)
        label = self.labels[idx]
        return {"input_values": spec, "labels": int(label)}
data_collator = DefaultDataCollator()
#metric = load_metric("accuracy")
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return metric.compute(predictions=predictions, references=labels)
training_args = TrainingArguments(
    output_dir="./ast-gtzan_w_pc",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=3e-5,
    num_train_epochs=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    gradient_accumulation_steps=8,
    greater_is_better=True,
    report_to="wandb",
    push_to_hub=True,
    hub_model_id="polinaZaroko/ast_try_again",
    hub_strategy="checkpoint",
    save_total_limit=2,
    warmup_ratio=0.1  #proportion of training to be dedicated to a linear warmup where learning rate gradually increases.
     
)
train_dataset = GTZANSpectrogramDataset(train, train_labels)
val_dataset = GTZANSpectrogramDataset(validation, validation_labels)

In [26]:
from transformers import PretrainedConfig
from transformers import ASTConfig
class ASTGenreConfig(ASTConfig):
    model_type= "ast-genre_classification"
    def __init__(self, num_labels = 10, **kwargs):
        super().__init__(**kwargs)
        self.num_labels = num_labels

In [27]:

ast_base = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

c:\Users\Kochana\projects\genres\ast_venv\Lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [28]:
from transformers import PreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput
import torch.nn as nn
import torch.nn.functional as F

class ASTForGenreClassification(PreTrainedModel):
    config_class = ASTGenreConfig

    def __init__(self, config, ast_model=ast_base):
        super().__init__(config)
        self.ast = ast_model
        self.classifier = nn.Linear(768, config.num_labels)
        self.dropout = nn.Dropout(0.2)

    def forward(self, input_values, labels=None):
        x = self.ast.embeddings(input_values)
        x = self.ast.encoder(x).last_hidden_state
        x = x.mean(dim=1)  # or use x[:, 0, :]
        x = self.dropout(x)
        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels, label_smoothing=0.1)

        return SequenceClassifierOutput(loss=loss, logits=logits)

In [17]:

model = ASTForGenreClassification.from_pretrained("polinaZaroko/ast_try_again")
model.eval()  # set to evaluation mode

c:\Users\Kochana\projects\genres\ast_venv\Lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


ASTForGenreClassification(
  (ast): ASTModel(
    (embeddings): ASTEmbeddings(
      (patch_embeddings): ASTPatchEmbeddings(
        (projection): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ASTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ASTLayer(
          (attention): ASTAttention(
            (attention): ASTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ASTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ASTIntermediate(
            (dense): Linear(in_features=7

In [23]:
test_metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
print(test_metrics)

100%|██████████| 50/50 [00:02<00:00, 21.17it/s]

{'test_loss': 1.3686814308166504, 'test_accuracy': 0.71, 'test_runtime': 2.7399, 'test_samples_per_second': 36.498, 'test_steps_per_second': 18.249}


In [29]:
config = ASTGenreConfig(num_labels=10)
ast_base = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
model = ASTForGenreClassification(config=config, ast_model=ast_base)

In [ ]:
from transformers.modeling_utils import load_state_dict
from transformers import PreTrainedModel
import torch.nn as nn
import torch.nn.functional as F
from transformers.utils import cached_file, WEIGHTS_NAME
from transformers.modeling_outputs import SequenceClassifierOutput

class ASTForGenreClassification(PreTrainedModel):
    config_class = ASTGenreConfig

    def __init__(self, config):
        super().__init__(config)
        self.ast = ASTModel(config)  # <-- instantiate from config, not passed in
        self.classifier = nn.Linear(768, config.num_labels)
        self.dropout = nn.Dropout(0.2)
        #self.post_init()  # ensure weights are initialized correctly
    
    def forward(self, input_values, labels=None):
        x = self.ast.embeddings(input_values)
        x = self.ast.encoder(x).last_hidden_state
        x = x.mean(dim=1)
        x = self.dropout(x)
        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels, label_smoothing=0.1)

        return SequenceClassifierOutput(loss=loss, logits=logits)

In [31]:
from torchinfo import summary
summary(model)

Layer (type:depth-idx)                                  Param #
ASTForGenreClassification                               --
├─ASTModel: 1-1                                         --
│    └─ASTEmbeddings: 2-1                               933,888
│    │    └─ASTPatchEmbeddings: 3-1                     197,376
│    │    └─Dropout: 3-2                                --
│    └─ASTEncoder: 2-2                                  --
│    │    └─ModuleList: 3-3                             85,054,464
│    └─LayerNorm: 2-3                                   1,536
├─Linear: 1-2                                           7,690
├─Dropout: 1-3                                          --
Total params: 86,194,954
Trainable params: 86,194,954
Non-trainable params: 0

In [31]:
import wandb
wandb.init(project="ast_model", name="wadims_pc_full", config={
            "epochs": training_args.num_train_epochs,
    "batch_size": training_args.per_device_train_batch_size,
    "lr": training_args.learning_rate,
    "model": "AST",
    "augmentation": False,
    "early_stopping" :8
   })

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/accuracy,▁▇▄█
eval/loss,█▂▇▁
eval/runtime,▆▁█▇
eval/samples_per_second,▃█▁▂
eval/steps_per_second,▃█▁▂
test/accuracy,▁
test/loss,▁
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
train/epoch,▁▁███▁▁███


In [32]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=None,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=8)],
)
#trainer.train()

c:\Users\Kochana\projects\genres\ast_venv\Lib\site-packages\accelerate\accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
c:\Users\Kochana\projects\genres\ast_venv\Lib\site-packages\accelerate\accelerator.py:463: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
trainer.train()

  2%|▏         | 45/2250 [00:41<33:20,  1.10it/s]

{'loss': 2.2908, 'grad_norm': 10.403141975402832, 'learning_rate': 5.733333333333334e-06, 'epoch': 1.0}



  2%|▏         | 45/2250 [00:46<33:20,  1.10it/s]Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 1024}


{'eval_loss': 2.0383145809173584, 'eval_accuracy': 0.38333333333333336, 'eval_runtime': 4.2384, 'eval_samples_per_second': 42.469, 'eval_steps_per_second': 21.235, 'epoch': 1.0}


  4%|▍         | 90/2250 [01:28<32:46,  1.10it/s]  

...

{'eval_loss': 1.4380508661270142, 'eval_accuracy': 0.5944444444444444, 'eval_runtime': 4.2264, 'eval_samples_per_second': 42.589, 'eval_steps_per_second': 21.295, 'epoch': 3.0}


  8%|▊         | 180/2250 [03:02<31:22,  1.10it/s]  

{'loss': 1.1947, 'grad_norm': 16.78443717956543, 'learning_rate': 2.3733333333333332e-05, 'epoch': 4.0}



  8%|▊         | 180/2250 [03:06<31:22,  1.10it/s]Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 1024}


{'eval_loss': 1.3059415817260742, 'eval_accuracy': 0.6277777777777778, 'eval_runtime': 4.2352, 'eval_samples_per_second': 42.501, 'eval_steps_per_second': 21.25, 'epoch': 4.0}


In [22]:
torch.save(model.state_dict(), "model_weights.pt")
torch.save(model.config, "config.pt")

In [24]:
config = torch.load("config.pt", weights_only=False)
ast_model = ASTModel(config)
model = ASTForGenreClassification(config, ast_model)
model.load_state_dict(torch.load("model_weights.pt"))
#model.eval()

<All keys matched successfully>

In [26]:
spec = data['gtzan_blues_00000/track']

In [28]:
model.eval()
with torch.no_grad():
    patches = model.ast.embeddings(spec)
    outputs = model.encoder(patches, output_hidden_states=True)
    last_hidden = outputs.hidden_states[-1]  # (1, num_patches, hidden_dim)
    embedding= last_hidden.mean(dim = 1)


AttributeError: 'numpy.ndarray' object has no attribute 'unsqueeze'

In [17]:
wandb.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/accuracy,▁█
eval/loss,█▁
eval/runtime,█▁
eval/samples_per_second,▁▁
eval/steps_per_second,▁▁
train/epoch,▁▁██
train/global_step,▁▁██
train/grad_norm,▁█
train/learning_rate,▁█
train/loss,█▁
eval/accuracy,0.51111


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.memory_summary(device=0))

True
NVIDIA GeForce RTX 4070 SUPER
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   1007 MiB |   5052 MiB | 233588 GiB | 233587 GiB |
|       from large pool |   1003 MiB |   5045 MiB | 232178 GiB | 232177 GiB |
|       from small pool |      3 MiB |      8 MiB |   1409 GiB |   1409 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   1007 MiB |   5052 MiB | 233588 GiB | 233587 GiB |
|       from large pool |   1

In [ ]:
print(torch.cuda.is_available())

True


In [ ]:
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [ ]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

^C
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Zugriff verweigert: 'C:\\Users\\Kochana\\projects\\genres\\ast_venv\\Lib\\site-packages\\~orch\\lib\\asmjit.dll'
Check the permissions.


[notice] A new release of pip available: 22.3 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cu126
  Obtaining dependency information for torchvision from https://download.pytorch.org/whl/cu126/torchvision-0.22.0%2Bcu126-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torchaudio from https://download.pytorch.org/whl/cu126/torchaudio-2.7.0%2Bcu126-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torch from https://download.pytorch.org/whl/cu126/torch-2.7.0%2Bcu126-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for pillow!=8.3.*,>=5.3.0 from https://download.pytorch.org/whl/pillow-11.0.0-cp311-cp311-win_amd64.whl.metadata
  Using cached https://download.pytorch.org/whl/pillow-11.0.0-cp311-cp311-win_amd64.whl.metadata (9.3 kB)
   ---------------------------------------- 6.3/6.3 MB 6.6 MB/s eta 0:00:00
   ---------------------------------------- 2.8/2.8 GB 994.8 kB/s eta 0:00:00
   ---------------------------------------- 4.2/4.2 MB 7.1 MB/s eta 0:00:00
  

In [ ]:
import torch
print(torch.__version__)

2.7.0+cpu


In [ ]:
pip install accelerate==0.28.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
best_hyp =  {'num_layers': 2, 'drop_out0': 2, 'dim_0': 128, 'drop_out1': 2, 'dim_1': 128, 'drop_out2': 1, 'dim_2': 128, 'drop_out3': 3, 'dim_3': 256, 'dim_last': 16}
df = pd.DataFrame(best_hyp, index = ['i',])
df.to_csv("best_hyp.csv")

In [1]:
a = [1,2,3]
b = [11] +a
b

[11, 1, 2, 3]